In [1]:
from pathlib import Path
import hashlib
import warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from arch.bootstrap import optimal_block_length

input_file = Path("SP500_daily_forecasts.csv")
output_directory = Path("predictive_out_of_sample_test")
training_observations = 1260
minimum_breaches = 10
minimum_nonbreaches = 10
refit_every = 21
bootstrap_replications = 2000
seed = 20261025
tail_probabilities = [0.01, 0.03, 0.05]
forecast_horizons = [1, 5, 10, 21]
features = {
    "VER": ["VaR_Ratio"],
    "Volatility": ["Sigma"],
    "VER plus volatility": ["VaR_Ratio", "Sigma"],
}
benchmarks = [
    "Target tail probability",
    "Historical breach frequency",
]
epsilon = 1e-12
horizon_label = "Forecast horizon (trading days)"


def load_data():
    data = pd.read_csv(input_file)
    required = [
        "RollingEnd",
        "VaR_Estimation_Date",
        "Alpha",
        "ForecastHorizon",
        "VaR_Ratio",
        "Sigma",
        "Breach_FHS",
    ]
    missing = sorted(set(required) - set(data.columns))

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for column in required[:2]:
        data[column] = pd.to_datetime(
            data[column], errors="raise"
        ).dt.tz_localize(None)

    for column in required[2:]:
        data[column] = pd.to_numeric(data[column], errors="raise")

    if data[required].isna().any().any():
        raise ValueError("Required fields contain missing values.")

    if not np.isfinite(
        data[required[2:]].to_numpy(dtype=float)
    ).all():
        raise ValueError("Required numeric fields contain nonfinite values.")

    if not data["Breach_FHS"].isin([0, 1]).all():
        raise ValueError("Breach indicators must equal zero or one.")

    if not data["Sigma"].gt(0).all():
        raise ValueError("Volatility must be positive.")

    if not (
        data["RollingEnd"] < data["VaR_Estimation_Date"]
    ).all():
        raise ValueError("Evaluation dates must follow forecast origins.")

    if data.duplicated(
        ["RollingEnd", "Alpha", "ForecastHorizon"]
    ).any():
        raise ValueError("Duplicate origins within a tail/horizon specification.")

    selected = np.isclose(
        data["Alpha"].to_numpy()[:, None],
        np.asarray(tail_probabilities)[None, :],
    ).any(axis=1)

    data = data.loc[
        selected & data["ForecastHorizon"].isin(forecast_horizons)
    ].copy()

    if data.empty:
        raise ValueError("No observations match the requested specifications.")

    return data.sort_values(
        ["Alpha", "ForecastHorizon", "RollingEnd"]
    ).reset_index(drop=True)


def generate_predictions(data):
    records = []

    for alpha in tail_probabilities:
        for horizon in forecast_horizons:
            subset = data.loc[
                np.isclose(data["Alpha"], alpha)
                & data["ForecastHorizon"].eq(horizon)
            ].sort_values("RollingEnd").reset_index(drop=True)

            if subset.empty:
                continue

            if not subset["VaR_Estimation_Date"].is_monotonic_increasing:
                raise ValueError("Evaluation dates must increase within each specification.")

            fitted = None
            last_fit = -refit_every
            training = None
            fitted_at = None
            historical_probability = np.nan

            for position, observation in subset.iterrows():
                origin = observation["RollingEnd"]
                record = {
                    "Tail probability": alpha,
                    horizon_label: horizon,
                    "Forecast origin": origin,
                    "Evaluation date": observation["VaR_Estimation_Date"],
                    "Actual breach": int(observation["Breach_FHS"]),
                }

                if fitted is None or position - last_fit >= refit_every:
                    past = subset.loc[
                        subset["RollingEnd"].lt(origin)
                        & subset["VaR_Estimation_Date"].le(origin)
                    ].tail(training_observations)

                    if len(past) < training_observations:
                        records.append({
                            **record,
                            "Status": "Insufficient completed training observations",
                        })
                        fitted = None
                        continue

                    breach_count = int(past["Breach_FHS"].sum())
                    nonbreach_count = len(past) - breach_count

                    if (
                        breach_count < minimum_breaches
                        or nonbreach_count < minimum_nonbreaches
                    ):
                        records.append({
                            **record,
                            "Status": "Insufficient training breaches or non-breaches",
                        })
                        fitted = None
                        continue

                    candidates = {}

                    try:
                        for name, columns in features.items():
                            model = make_pipeline(
                                StandardScaler(),
                                LogisticRegression(
                                    C=1.0,
                                    penalty="l2",
                                    solver="lbfgs",
                                    max_iter=2000,
                                ),
                            )

                            with warnings.catch_warnings(record=True) as caught:
                                warnings.simplefilter("always")
                                model.fit(
                                    past[columns],
                                    past["Breach_FHS"].astype(int),
                                )

                            if any(
                                "converg" in str(item.message).lower()
                                for item in caught
                            ):
                                raise RuntimeError("Logistic optimizer convergence warning.")

                            candidates[name] = model

                    except Exception as exc:
                        records.append({
                            **record,
                            "Status": f"Fit unavailable: {type(exc).__name__}: {exc}",
                        })
                        fitted = None
                        continue

                    fitted = candidates
                    training = past
                    last_fit = position
                    fitted_at = origin
                    historical_probability = (
                        breach_count + 0.5
                    ) / (len(training) + 1)

                record.update({
                    "Status": "Estimated",
                    "Last model fitting origin": fitted_at,
                    "Latest training outcome date": training[
                        "VaR_Estimation_Date"
                    ].max(),
                    "Training observations": len(training),
                    "Training breaches": int(training["Breach_FHS"].sum()),
                    "Target tail probability": alpha,
                    "Historical breach frequency": historical_probability,
                })

                for name, columns in features.items():
                    record[name] = float(
                        fitted[name].predict_proba(
                            subset.loc[[position], columns]
                        )[0, 1]
                    )

                records.append(record)

            print(
                f"Tail probability {alpha:.0%}, horizon {horizon}: predictions completed",
                flush=True,
            )

    return pd.DataFrame(records)


def calculate_metrics(outcomes, probabilities):
    outcomes = np.asarray(outcomes, dtype=float)
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        epsilon,
        1 - epsilon,
    )

    return {
        "Brier score": float(np.mean((probabilities - outcomes) ** 2)),
        "Log loss": float(np.mean(
            -outcomes * np.log(probabilities)
            - (1 - outcomes) * np.log1p(-probabilities)
        )),
        "AUC": (
            float(roc_auc_score(outcomes, probabilities))
            if np.unique(outcomes).size == 2 else np.nan
        ),
        "Calibration gap": float(probabilities.mean() - outcomes.mean()),
    }


def calculate_calibration(outcomes, probabilities):
    outcomes = np.asarray(outcomes, dtype=float)
    predictor = logit(np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    ))

    if np.unique(outcomes).size < 2:
        return np.nan, np.nan, "No outcome variation"

    if np.std(predictor) < 1e-10:
        return np.nan, np.nan, "Constant forecast; slope not identifiable"

    def objective(parameters):
        linear_predictor = parameters[0] + parameters[1] * predictor
        return np.mean(
            np.logaddexp(0, linear_predictor)
            - outcomes * linear_predictor
        )

    result = minimize(objective, [0.0, 1.0], method="BFGS")

    if not result.success or not np.isfinite(result.x).all():
        return np.nan, np.nan, "Calibration fit did not converge"

    return (
        float(result.x[0]),
        float(result.x[1]),
        "Descriptive evaluation-sample calibration",
    )


def draw_block_indices(dates, length, generator):
    dates = pd.DatetimeIndex(dates)
    differences = np.diff(
        dates.values
    ).astype("timedelta64[D]").astype(int)
    gaps = np.flatnonzero(differences > 7) + 1
    starts = np.r_[0, gaps]
    ends = np.r_[gaps, len(dates)]
    pieces = []

    for start, end in zip(starts, ends):
        segment_size = end - start
        effective_length = min(length, segment_size)
        block_count = int(np.ceil(segment_size / effective_length))
        block_starts = generator.integers(
            0, segment_size, size=block_count
        )
        indices = (
            block_starts[:, None]
            + np.arange(effective_length)[None, :]
        ) % segment_size
        pieces.append(indices.ravel()[:segment_size] + start)

    return np.concatenate(pieces)


def percentile_interval(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan, 0

    return (
        float(np.quantile(values, 0.025)),
        float(np.quantile(values, 0.975)),
        len(values),
    )


def choose_block_lengths(outcomes, probabilities, pairs):
    series = {}

    for name, probability in probabilities.items():
        probability = np.clip(probability, epsilon, 1 - epsilon)
        series[f"{name}: Brier"] = (probability - outcomes) ** 2
        series[f"{name}: Log loss"] = (
            -outcomes * np.log(probability)
            - (1 - outcomes) * np.log1p(-probability)
        )

    for model, benchmark in pairs:
        for measure in ["Brier", "Log loss"]:
            series[f"{model} minus {benchmark}: {measure}"] = (
                series[f"{model}: {measure}"]
                - series[f"{benchmark}: {measure}"]
            )

    block_data = pd.DataFrame(series)
    block_data = block_data.loc[:, block_data.std(ddof=0).gt(0)]

    if len(outcomes) < 25 or block_data.empty:
        return None, [], "Insufficient observations for block selection"

    automatic = optimal_block_length(
        block_data
    )["circular"].to_numpy(dtype=float)
    automatic = automatic[np.isfinite(automatic) & (automatic > 0)]

    if len(automatic) == 0:
        return None, [], "Automatic block selection unavailable"

    selected = int(np.ceil(automatic.max()))
    cap = max(1, len(outcomes) // 5)
    primary = min(max(5, selected), cap)
    lengths = sorted(set([
        max(1, primary // 2),
        primary,
        min(cap, 2 * primary),
    ]))
    status = (
        "Automatic length capped; interpret intervals cautiously"
        if selected > cap
        else "Automatic loss-based selection with sensitivity lengths"
    )

    return (selected, primary), lengths, status


def evaluate_predictions(predictions):
    summaries, comparisons, settings = [], [], []
    names = benchmarks + list(features)
    metric_names = ["Brier score", "Log loss", "AUC", "Calibration gap"]
    pairs = [
        ("VER", benchmark) for benchmark in benchmarks
    ] + [("VER plus volatility", "Volatility")]

    for alpha_index, alpha in enumerate(tail_probabilities):
        for horizon in forecast_horizons:
            raw = predictions.loc[
                np.isclose(predictions["Tail probability"], alpha)
                & predictions[horizon_label].eq(horizon)
            ]
            subset = raw.loc[
                raw["Status"].eq("Estimated")
            ].sort_values("Forecast origin")
            base = {
                "Tail probability": alpha,
                horizon_label: horizon,
            }

            if subset.empty:
                for name in names:
                    summaries.append({
                        **base,
                        "Model": name,
                        "Observations": 0,
                        "Status": "No eligible out-of-sample predictions",
                    })
                continue

            outcomes = subset["Actual breach"].to_numpy(dtype=float)
            probabilities = {
                name: subset[name].to_numpy(dtype=float)
                for name in names
            }
            points = {
                name: calculate_metrics(outcomes, probabilities[name])
                for name in names
            }
            calibrations = {
                name: calculate_calibration(outcomes, probabilities[name])
                for name in names
            }

            selection, lengths, status = choose_block_lengths(
                outcomes, probabilities, pairs
            )

            if selection is None:
                settings.append({
                    **base,
                    "Observations": len(subset),
                    "Status": status,
                })

                for name in names:
                    intercept, slope, calibration_status = calibrations[name]
                    summaries.append({
                        **base,
                        "Model": name,
                        "Observations": len(subset),
                        "Breaches": int(outcomes.sum()),
                        **points[name],
                        "Calibration intercept": intercept,
                        "Calibration slope": slope,
                        "Calibration status": calibration_status,
                        "Status": status,
                    })
                continue

            automatic_length, primary_length = selection

            settings.append({
                **base,
                "Observations": len(subset),
                "Automatic circular block length": automatic_length,
                "Primary block length": primary_length,
                "Sensitivity block lengths": ",".join(map(str, lengths)),
                "Bootstrap replications": bootstrap_replications,
                "Status": status,
            })

            for length in lengths:
                draws = {
                    name: {metric: [] for metric in metric_names}
                    for name in names
                }
                generator = np.random.default_rng(
                    np.random.SeedSequence([
                        seed, alpha_index, horizon, length
                    ])
                )

                for _ in range(bootstrap_replications):
                    indices = draw_block_indices(
                        subset["Forecast origin"], length, generator
                    )

                    for name in names:
                        estimates = calculate_metrics(
                            outcomes[indices],
                            probabilities[name][indices],
                        )

                        for metric, value in estimates.items():
                            draws[name][metric].append(value)

                for name in names:
                    intercept, slope, calibration_status = calibrations[name]
                    record = {
                        **base,
                        "Model": name,
                        "Observations": len(subset),
                        "Breaches": int(outcomes.sum()),
                        "First forecast origin": subset["Forecast origin"].min(),
                        "Last forecast origin": subset["Forecast origin"].max(),
                        "First evaluation date": subset["Evaluation date"].min(),
                        "Last evaluation date": subset["Evaluation date"].max(),
                        "Block length": length,
                        "Primary block specification": length == primary_length,
                        "Mean predicted probability": float(
                            probabilities[name].mean()
                        ),
                        "Observed breach frequency": float(outcomes.mean()),
                        "Calibration intercept": intercept,
                        "Calibration slope": slope,
                        "Calibration status": calibration_status,
                        "Skipped origins": int(
                            raw["Status"].ne("Estimated").sum()
                        ),
                        "Status": "Evaluated",
                    }

                    for metric, value in points[name].items():
                        lower, upper, valid_draws = percentile_interval(
                            draws[name][metric]
                        )
                        record.update({
                            metric: value,
                            f"{metric} 95% lower": lower,
                            f"{metric} 95% upper": upper,
                            f"{metric} valid bootstrap draws": valid_draws,
                        })

                    summaries.append(record)

                for model, benchmark in pairs:
                    for metric in metric_names:
                        differences = (
                            np.asarray(draws[model][metric])
                            - np.asarray(draws[benchmark][metric])
                        )
                        lower, upper, valid_draws = percentile_interval(
                            differences
                        )

                        comparisons.append({
                            **base,
                            "Model": model,
                            "Benchmark": benchmark,
                            "Metric": metric,
                            "Observations": len(subset),
                            "Block length": length,
                            "Primary block specification": length == primary_length,
                            "Difference (model minus benchmark)": (
                                points[model][metric]
                                - points[benchmark][metric]
                            ),
                            "95% lower": lower,
                            "95% upper": upper,
                            "Valid bootstrap draws": valid_draws,
                        })

                print(
                    f"Tail probability {alpha:.0%}, horizon {horizon}, "
                    f"block length {length}: evaluation completed",
                    flush=True,
                )

    return (
        pd.DataFrame(summaries),
        pd.DataFrame(comparisons),
        pd.DataFrame(settings),
    )


def main():
    output_directory.mkdir(parents=True, exist_ok=True)
    data = load_data()

    configuration = {
        "input_hash": hashlib.sha256(input_file.read_bytes()).hexdigest(),
        "training_observations": training_observations,
        "minimum_breaches": minimum_breaches,
        "minimum_nonbreaches": minimum_nonbreaches,
        "refit_every": refit_every,
        "tail_probabilities": tail_probabilities,
        "forecast_horizons": forecast_horizons,
        "features": features,
        "ridge_c": 1.0,
        "version": 3,
    }
    cache_path = output_directory / "prediction_cache.pkl"

    if cache_path.exists():
        stored = pd.read_pickle(cache_path)
        predictions = (
            stored["predictions"]
            if stored["configuration"] == configuration
            else generate_predictions(data)
        )
    else:
        predictions = generate_predictions(data)

    if predictions.empty:
        raise RuntimeError("No forecast records were generated.")

    pd.to_pickle(
        {
            "configuration": configuration,
            "predictions": predictions,
        },
        cache_path,
    )

    valid = predictions.loc[predictions["Status"].eq("Estimated")]

    if not valid.empty:
        if not (
            valid["Latest training outcome date"]
            <= valid["Last model fitting origin"]
        ).all():
            raise RuntimeError("Training outcome timing violation.")

        if not (
            valid["Last model fitting origin"]
            <= valid["Forecast origin"]
        ).all():
            raise RuntimeError("Model fitting timing violation.")

        if not (
            valid["Forecast origin"] < valid["Evaluation date"]
        ).all():
            raise RuntimeError("Forecast evaluation timing violation.")

    predictions.to_csv(
        output_directory / "OOS_predictions.csv",
        index=False,
        na_rep="N/A",
    )

    summaries, comparisons, settings = evaluate_predictions(predictions)

    summaries.to_csv(
        output_directory / "OOS_metrics_with_intervals.csv",
        index=False,
        na_rep="N/A",
    )
    comparisons.to_csv(
        output_directory / "OOS_paired_comparisons.csv",
        index=False,
        na_rep="N/A",
    )
    settings.to_csv(
        output_directory / "Dependence_settings.csv",
        index=False,
        na_rep="N/A",
    )

    print(f"Results saved to {output_directory.resolve()}")


if __name__ == "__main__":
    main()

Tail probability 1%, horizon 1: predictions completed
Tail probability 1%, horizon 5: predictions completed
Tail probability 1%, horizon 10: predictions completed
Tail probability 1%, horizon 21: predictions completed
Tail probability 3%, horizon 1: predictions completed
Tail probability 3%, horizon 5: predictions completed
Tail probability 3%, horizon 10: predictions completed
Tail probability 3%, horizon 21: predictions completed
Tail probability 5%, horizon 1: predictions completed
Tail probability 5%, horizon 5: predictions completed
Tail probability 5%, horizon 10: predictions completed
Tail probability 5%, horizon 21: predictions completed
Tail probability 1%, horizon 1, block length 4: evaluation completed
Tail probability 1%, horizon 1, block length 8: evaluation completed
Tail probability 1%, horizon 1, block length 16: evaluation completed
Tail probability 1%, horizon 5, block length 16: evaluation completed
Tail probability 1%, horizon 5, block length 32: evaluation complete